# Day 038 — Exercise 4: pivot_summary

**What you'll build:** `pivot_summary(df, index, columns, values, aggfunc='mean') -> pd.DataFrame` — wrap `pd.pivot_table` to produce a clean cross-tabulation with no NaN values (`fill_value=0`) and no column axis name artefact.

**Why it matters:** Pivot tables turn a long-format DataFrame into a readable matrix — product × region revenue, campaign × channel clicks, user × feature usage. They are the standard summary a stakeholder wants to see.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# SALES_DF: 8 rows, 6 columns
# product:  Widget×4, Gadget×2, Doohickey×2
# revenue   = price × quantity  (pre-computed)
SALES_DF = pd.DataFrame({
    'product':  ['Widget', 'Widget', 'Widget', 'Widget',
                 'Gadget', 'Gadget', 'Doohickey', 'Doohickey'],
    'category': ['Elec', 'Elec', 'Elec', 'Elec',
                 'Elec', 'Elec', 'Access', 'Access'],
    'region':   ['North', 'South', 'East', 'West',
                 'North', 'East', 'North', 'South'],
    'price':    [25.0, 25.0, 25.0, 25.0, 150.0, 150.0, 8.0, 8.0],
    'quantity': [10, 5, 4, 6, 3, 7, 50, 15],
    'revenue':  [250.0, 125.0, 100.0, 150.0, 450.0, 1050.0, 400.0, 120.0],
})

import pandas as pd

def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count':      int(s.count()),
            'mean':       round(float(s.mean()), 4),
            'std':        round(float(s.std()), 4),
            'min':        float(s.min()),
            'q25':        float(s.quantile(0.25)),
            'median':     float(s.quantile(0.50)),
            'q75':        float(s.quantile(0.75)),
            'max':        float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count':      int(s.count()),
        'unique':     int(s.nunique()),
        'top':        str(counts.index[0]) if len(counts) else None,
        'top_freq':   int(counts.iloc[0])  if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }

import pandas as pd

def top_groups(df: pd.DataFrame, group_col: str, value_col: str,
               n: int = 5) -> pd.DataFrame:
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )

import pandas as pd

def correlation_summary(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    corr   = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)

## Your Implementation

In [ ]:
def pivot_summary(df: pd.DataFrame, index: str, columns: str,
                  values: str, aggfunc: str = 'mean') -> pd.DataFrame:
    """
    Create a pivot table and return it as a plain DataFrame.

    Args:
        index   — column whose values become row labels
        columns — column whose values become column headers
        values  — column to aggregate
        aggfunc — aggregation function string ('sum', 'mean', 'count')
    fill_value=0 so missing combinations are 0, not NaN.
    """
    # TODO: piv = pd.pivot_table(df, values=values, index=index,
    #                            columns=columns, aggfunc=aggfunc, fill_value=0)
    # TODO: piv.columns.name = None  (clear the columns axis name)
    # TODO: return piv.reset_index()
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns DataFrame
    try:
        assert 'pivot_summary' in globals()
        result = pivot_summary(SALES_DF, 'product', 'region', 'revenue', 'sum')
        assert isinstance(result, pd.DataFrame), \
            f'expected DataFrame, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: index column present and region columns present
    # regions in data: North, South, East, West
    try:
        result = pivot_summary(SALES_DF, 'product', 'region', 'revenue', 'sum')
        assert 'product' in result.columns, \
            f"'product' column missing; columns={list(result.columns)}"
        for reg in ('North', 'South', 'East', 'West'):
            assert reg in result.columns, \
                f'{reg!r} column missing; columns={list(result.columns)}'
        passed += 1; print('\u2705 Check 2: product + all 4 region columns present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: correct value for Widget-North (price=25, quantity=10 → revenue=250)
    try:
        result = pivot_summary(SALES_DF, 'product', 'region', 'revenue', 'sum')
        widget_row = result[result['product'] == 'Widget'].iloc[0]
        assert float(widget_row['North']) == 250.0, \
            f"Widget-North should be 250, got {widget_row['North']}"
        assert float(widget_row['South']) == 125.0, \
            f"Widget-South should be 125, got {widget_row['South']}"
        passed += 1; print('\u2705 Check 3: Widget-North=250, Widget-South=125')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: fill_value=0 — no NaN in result (Gadget has no South/West data)
    try:
        result = pivot_summary(SALES_DF, 'product', 'region', 'revenue', 'sum')
        null_total = result.isnull().sum().sum()
        assert null_total == 0, \
            f'{null_total} NaN values — did you use fill_value=0?'
        gadget_row  = result[result['product'] == 'Gadget'].iloc[0]
        assert float(gadget_row['South']) == 0.0, \
            f'Gadget-South should be 0 (no data), got {gadget_row["South"]}'
        passed += 1; print('\u2705 Check 4: no NaN; Gadget-South=0 (fill_value=0)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: aggfunc='count' gives row counts instead of sums
    try:
        cnt = pivot_summary(SALES_DF, 'product', 'region', 'revenue', 'count')
        widget_row = cnt[cnt['product'] == 'Widget'].iloc[0]
        # Widget has 1 row per region (North, South, East, West)
        for reg in ('North', 'South', 'East', 'West'):
            assert float(widget_row[reg]) == 1.0, \
                f'Widget-{reg} count should be 1, got {widget_row[reg]}'
        passed += 1; print('\u2705 Check 5: aggfunc=count gives per-cell row counts')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def pivot_summary(df: pd.DataFrame, index: str, columns: str,
                  values: str, aggfunc: str = 'mean') -> pd.DataFrame:
    piv = pd.pivot_table(
        df, values=values, index=index, columns=columns,
        aggfunc=aggfunc, fill_value=0,
    )
    piv.columns.name = None
    return piv.reset_index()
```

</details>